In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path


In [ ]:
df = pd.read_csv('data/data.csv',encoding='latin-1')
df = df.drop_duplicates()
service_codes = ['POST','DOT','M','BANK CHARGES','AMAZONFEE','CRUK','D','S']
df = df[~df['StockCode'].isin(service_codes)]
print('Dataset info:')
display(df.head())
df.info()


In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('Duplicate values:')
print(df.duplicated().sum())

In [ ]:
df = df.dropna(subset=['CustomerID'])
df = df[df['UnitPrice'] > 0]
df = df[df['Quantity'] > 0]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df['revenue'] = df['Quantity'] * df['UnitPrice']

In [ ]:
customer_count = df['CustomerID'].nunique()
total_products = df['StockCode'].nunique()
total_orders = df['InvoiceNo'].nunique()
total_revenue = df['revenue'].sum().round(2)
avg_check = df.groupby('InvoiceNo')['revenue'].sum().mean()

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Customers',
        'Orders',
        'Products',
        'Revenue',
        'Average Check'
    ],
    'Value': [
        customer_count,
        total_orders,
        total_products,
        total_revenue,
        round(avg_check,2)
    ]
})

display(summary)

In [ ]:
# новый чистый датафрейм
clean = df.copy()


In [ ]:
# временные тренды по месяцам 
sales_by_month = clean.groupby(clean['InvoiceDate'].dt.to_period('M')).agg({
    'InvoiceNo': 'nunique',
    'revenue': 'sum'
}).rename(columns={'InvoiceDate': 'Month', 'InvoiceNo': 'Orders', 'revenue': 'Revenue'}).reset_index(names='Month')

display(sales_by_month)



In [ ]:
sales_by_weekday = clean.groupby(clean['InvoiceDate'].dt.day_name()).agg({
    'InvoiceNo': 'nunique',
    'revenue': 'sum'
}).rename(columns={'InvoiceNo': 'Orders', 'revenue': 'Revenue'}).reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
]).reset_index(names='Weekday')


display(sales_by_weekday)

In [ ]:
country_summary = (
    clean.groupby('Country')
    .agg(
        Revenue=('revenue', 'sum'),
        Orders=('InvoiceNo', 'nunique'),
        Customers=('CustomerID', 'nunique'),
        Quantity=('Quantity', 'sum'),
    )
    .reset_index()
    .sort_values('Revenue', ascending=False)
    .round(2)
)

display(country_summary)

In [ ]:
# top products by sales
products_by_sales = (
    clean
    .groupby('Description')
    .agg(
        Quantity=('Quantity', 'sum'),
        Orders=('InvoiceNo', 'nunique')
    )
    .sort_values('Quantity', ascending=False)
    .reset_index()
)

display(products_by_sales)

In [ ]:
# top products by revenue
top_products_by_revenue = clean.groupby('Description').agg({
    'InvoiceNo': 'nunique',
    'revenue': 'sum'
}).rename(columns={'InvoiceNo': 'Orders', 'revenue': 'Revenue'}).sort_values(by='Revenue', ascending=False).reset_index().round(2)
display(top_products_by_revenue)

In [ ]:
product_summary = (
    clean
    .groupby('Description')
    .agg(
        Quantity=('Quantity', 'sum'),
        Revenue=('revenue', 'sum'),
        Orders=('InvoiceNo', 'nunique')
    )
    .reset_index()
)

In [ ]:
snap_date = df['InvoiceDate'].max() + pd.DateOffset(days=1)

recency = (
    snap_date -
    df.groupby('CustomerID')['InvoiceDate'].max()
).dt.days


frequency = df.groupby('CustomerID')['InvoiceNo'].nunique()
monetary = df.groupby('CustomerID')['revenue'].sum()

rfm = pd.DataFrame({
    'recency': recency,
    'frequency': frequency,
    'monetary': monetary
})

print('RFM table:')
display(rfm.head())


In [ ]:
#scores

rfm['R_score'] = pd.qcut(
    rfm['recency'],
    5,
    labels=[5, 4, 3, 2, 1]

)

rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
)



rfm['RFM'] = (
    rfm['R_score'].astype(str)
    + rfm['F_score'].astype(str)
    + rfm['M_score'].astype(str)

)

In [ ]:
def segment(row):
    rules = [
        (row == '555', 'Champions'),
        (row.startswith('55'), 'Loyal'),
        (row.startswith('5'), 'Recent'),
        (row.endswith('55'), 'Big Spenders'),
        (row == '111', 'Lost'),
    ]

    return next(
        (label for condition, label in rules if condition),
        'Others'
    )

rfm['Segment'] = rfm['RFM'].apply(segment)

In [ ]:
rfm.groupby('Segment').agg({
    'recency': ['mean', 'median'],
    'frequency': ['mean', 'median'],
    'monetary': ['mean', 'median', 'sum']
})

In [ ]:
rfm['Segment'].value_counts(normalize=True) * 100

In [ ]:
# segment size
segment_size = rfm.groupby('Segment').agg({
    'recency': 'count'
}).rename(columns={'recency': 'Count'}).sort_values(by='Count', ascending=False).reset_index()
display(segment_size)

In [ ]:
customer_summary = (
    clean.groupby('CustomerID')
    .agg(
        Revenue=('revenue', 'sum'),
        Orders=('InvoiceNo', 'nunique'),
        Quantity=('Quantity', 'sum'),
    )
    .reset_index()
    .sort_values('Revenue', ascending=False)
    .round(2)
)

display(customer_summary)


In [ ]:
from pathlib import Path

Path('data').mkdir(exist_ok=True)

clean.to_csv('data/clean.csv', index=False)

summary.to_csv('data/kpi.csv', index=False)

sales_by_month.to_csv('data/sales_by_month.csv', index=False)
sales_by_weekday.to_csv('data/sales_by_weekday.csv', index=False)
product_summary.to_csv('data/product_summary.csv', index=False)
products_by_sales.to_csv('data/products_by_sales.csv', index=False)
top_products_by_revenue.to_csv('data/products_by_revenue.csv', index=False)
country_summary.to_csv('data/country_summary.csv', index=False)
customer_summary.to_csv('data/customer_summary.csv', index=False)
rfm.to_csv('data/rfm.csv', index=False)

segment_size.to_csv('data/segment_size.csv', index=False)
